# PhilWeather Analytics — Phase 2: Data Exploration & Profiling

## Overview
This notebook performs exploratory data analysis (EDA) on the raw 10-year historical Philippine weather dataset (`daily_data_combined_2010_to_2019.csv`) and its corresponding unit metadata (`daily_units_2010_to_2019.csv`).

### Objectives
1. **Load and map unit metadata** to DataFrame columns.
2. **Validate data integrity** (missing values, data types, value ranges, and duplicate records).
3. **Analyze statistical distribution** of key weather metrics (temperature, precipitation, wind speeds).
4. **Finalize schema design** for the upcoming PostgreSQL loading phase.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define paths relative to notebook location
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data" / "raw"
DATA_FILE = DATA_DIR / "daily_data_combined_2010_to_2019.csv"
UNITS_FILE = DATA_DIR / "daily_units_2010_to_2019.csv"

print(f"Data Directory: {DATA_DIR}")

## 1. Load Unit Metadata

In [ ]:
units_df = pd.read_csv(UNITS_FILE)
unit_map = units_df.to_dict(orient="records")[0]

print("=== UNIT METADATA MAPPING ===")
unit_summary = pd.DataFrame(list(unit_map.items()), columns=["Field Name", "Unit"])
display(unit_summary)

## 2. Load Raw Weather Data & Perform Data Quality Audit

In [ ]:
df = pd.read_csv(DATA_FILE)

print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {df.shape[1]}")
print(f"Total Missing/Null Values: {df.isnull().sum().sum()}")

duplicates = df.duplicated(subset=["city_name", "datetime"]).sum()
print(f"Duplicates on (city_name, datetime): {duplicates}")

## 3. Geographical & Temporal Coverage

In [ ]:
cities = df["city_name"].nunique()
min_date = df["datetime"].min()
max_date = df["datetime"].max()

print(f"Unique Cities: {cities}")
print(f"Start Date: {min_date}")
print(f"End Date: {max_date}")
print(f"Days per City: {df.groupby('city_name')['datetime'].count().iloc[0]}")

## 4. Meteorological Metric Statistics

In [ ]:
weather_metrics = [
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean",
    "precipitation_sum", "rain_sum", "snowfall_sum", "precipitation_hours",
    "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum",
    "et0_fao_evapotranspiration"
]

summary = df[weather_metrics].describe().T[["min", "mean", "50%", "max", "std"]]
summary["unit"] = [unit_map.get(m, "N/A") for m in weather_metrics]
display(summary)

## 5. Summary of Key Findings & Schema Recommendations
- **Completeness**: 0 missing values across all 500,324 rows.
- **Key Constraints**: Unique composite key `(city_name, datetime)` holds perfectly across all 137 cities.
- **Redundant Columns**: `snowfall_sum` is uniformly `0.0` across the entire 10-year period (tropical context) and can be safely dropped during Phase 4 (Transform).
- **Duration Units**: `daylight_duration` and `sunshine_duration` are stored in seconds (`s`) and should be converted to hours (`h`) during transformation.